# Model Selection — Gurgaon House Price Prediction (Improved)

**Key fixes over original:**
- Fixed double-encoding bug in Strategy 2 (sector/agePossession encoded by both Ordinal AND OHE)
- `scorer()` defined once, accepts preprocessor as parameter — no copy-paste
- Added RMSE metric alongside MAE
- Added train R² to detect overfitting
- Replaced `Lasso()` (broken, R²=0.059) with `LassoCV` for auto alpha selection
- Removed deprecated `max_features='auto'` from GridSearch
- Tuned **XGBoost** (best performer) in addition to Random Forest
- Exported model is now the **best tuned model**, not a manually selected one

In [ ]:
# Installs
import subprocess
subprocess.run(['pip', 'install', 'xgboost', 'category_encoders', '-q'])

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import KFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
from xgboost import XGBRegressor
import category_encoders as ce

print("All libraries loaded.")

In [ ]:
# Load dataset
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

# Map furnishing codes to readable labels
df['furnishing_type'] = df['furnishing_type'].replace({
    0.0: 'unfurnished', 1.0: 'semifurnished', 2.0: 'furnished'
})

X = df.drop(columns=['price'])
y = df['price']

# Log-transform target (reduces skewness 3.28 to ~1.07)
y_transformed = np.log1p(y)

print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Price range: {y.min():.2f} to {y.max():.2f} Crores (mean: {y.mean():.2f})")

## Feature Groups

In [ ]:
NUM_COLS = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']

# Columns for ordinal encoding only (not OHE)
ORD_COLS_ONLY    = ['property_type', 'balcony', 'luxury_category', 'floor_category']

# Columns for OHE (high-cardinality or nominal) — excluded from ORD in Strategy 2+
OHE_COLS         = ['sector', 'agePossession', 'furnishing_type']

# All ordinal cols (Strategy 1 only)
ORD_COLS_ALL     = ['property_type', 'sector', 'balcony', 'agePossession',
                    'furnishing_type', 'luxury_category', 'floor_category']

KFOLD = KFold(n_splits=10, shuffle=True, random_state=42)
print("Feature groups defined.")

## Unified Scorer

Defined **once**, reused across all strategies. Reports:
- `CV R²` — 10-fold cross-validation (primary metric)
- `Test MAE` — Mean Absolute Error in Crores
- `Test RMSE` — Root Mean Squared Error in Crores  
- `Train R²` — to detect overfitting
- `Test R²` — held-out test set

In [ ]:
def scorer(model_name, model, preprocessor):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    cv_r2 = cross_val_score(pipeline, X, y_transformed, cv=KFOLD, scoring='r2').mean()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_transformed, test_size=0.2, random_state=42
    )
    pipeline.fit(X_train, y_train)

    y_pred      = np.expm1(pipeline.predict(X_test))
    y_test_orig = np.expm1(y_test)
    y_train_pred = np.expm1(pipeline.predict(X_train))
    y_train_orig = np.expm1(y_train)

    mae      = mean_absolute_error(y_test_orig, y_pred)
    rmse     = np.sqrt(mean_squared_error(y_test_orig, y_pred))
    test_r2  = 1 - np.sum((y_test_orig - y_pred)**2) / np.sum((y_test_orig - y_test_orig.mean())**2)
    train_r2 = 1 - np.sum((y_train_orig - y_train_pred)**2) / np.sum((y_train_orig - y_train_orig.mean())**2)

    return [model_name, round(cv_r2, 4), round(mae, 4), round(rmse, 4),
            round(train_r2, 4), round(test_r2, 4)]


def run_all_models(preprocessor):
    model_dict = {
        'Linear Regression': LinearRegression(),
        'Ridge'            : Ridge(),
        # FIX: LassoCV auto-tunes alpha — original Lasso() gave R²=0.059 due to default alpha=1
        'LASSO (CV)'       : LassoCV(cv=5, max_iter=10000),
        'SVR'              : SVR(),
        'Decision Tree'    : DecisionTreeRegressor(random_state=42),
        'Random Forest'    : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'Extra Trees'      : ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42),
        'AdaBoost'         : AdaBoostRegressor(random_state=42),
        'MLP'              : MLPRegressor(max_iter=500, random_state=42),
        'XGBoost'          : XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1, verbosity=0),
    }
    results = [scorer(name, model, preprocessor) for name, model in model_dict.items()]
    cols = ['Model', 'CV R2', 'Test MAE', 'Test RMSE', 'Train R2', 'Test R2']
    return pd.DataFrame(results, columns=cols).sort_values('Test MAE').reset_index(drop=True)

print("scorer() and run_all_models() defined.")

## Strategy 1 — Ordinal Encoding

All categoricals encoded with OrdinalEncoder. Baseline strategy.

In [ ]:
preprocessor_ordinal = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OrdinalEncoder(), ORD_COLS_ALL),
], remainder='passthrough')

results_ordinal = run_all_models(preprocessor_ordinal)
results_ordinal

## Strategy 2 — One-Hot Encoding (Bug Fixed)

**Original bug:** `sector`, `agePossession`, `furnishing_type` were in BOTH `OrdinalEncoder` and `OneHotEncoder` lists, creating duplicate/conflicting features.

**Fix:** These columns are encoded by OHE only; removed from the ordinal list.

In [ ]:
# FIX: ORD_COLS_ONLY excludes columns being OHE'd (no more double encoding)
preprocessor_ohe = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                           NUM_COLS),
    ('ord', OrdinalEncoder(),                           ORD_COLS_ONLY),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False), OHE_COLS),
], remainder='passthrough')

results_ohe = run_all_models(preprocessor_ohe)
results_ohe

## Strategy 3 — One-Hot Encoding + PCA (95% variance)

PCA after OHE for dimensionality reduction. Generally hurts tree-based models but helps linear ones.

In [ ]:
preprocessor_pca_base = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                           NUM_COLS),
    ('ord', OrdinalEncoder(),                           ORD_COLS_ONLY),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False), OHE_COLS),
], remainder='passthrough')

def scorer_pca(model_name, model):
    pipeline = Pipeline([
        ('preprocessor', preprocessor_pca_base),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    cv_r2 = cross_val_score(pipeline, X, y_transformed, cv=KFOLD, scoring='r2').mean()
    X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    y_test_orig = np.expm1(y_test)
    mae  = mean_absolute_error(y_test_orig, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
    return [model_name, round(cv_r2, 4), round(mae, 4), round(rmse, 4)]

model_dict_pca = {
    'Linear Regression': LinearRegression(),
    'Ridge'            : Ridge(),
    'LASSO (CV)'       : LassoCV(cv=5, max_iter=10000),
    'SVR'              : SVR(),
    'Decision Tree'    : DecisionTreeRegressor(random_state=42),
    'Random Forest'    : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees'      : ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost'         : AdaBoostRegressor(random_state=42),
    'MLP'              : MLPRegressor(max_iter=500, random_state=42),
    'XGBoost'          : XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1, verbosity=0),
}
results_pca = pd.DataFrame(
    [scorer_pca(n, m) for n, m in model_dict_pca.items()],
    columns=['Model', 'CV R2', 'Test MAE', 'Test RMSE']
).sort_values('Test MAE').reset_index(drop=True)
results_pca

## Strategy 4 — Target Encoding on Sector (Best Strategy)

`sector` (113 unique values) encoded with TargetEncoder — maps each sector to its mean target value. Avoids OHE explosion of 113 dummy columns while capturing geographic price signal better than ordinal.

In [ ]:
preprocessor_target = ColumnTransformer(transformers=[
    ('num',    StandardScaler(),                           NUM_COLS),
    ('ord',    OrdinalEncoder(),
               ['property_type', 'balcony', 'furnishing_type', 'luxury_category', 'floor_category']),
    ('ohe',    OneHotEncoder(drop='first', sparse_output=False), ['agePossession']),
    ('target', ce.TargetEncoder(),                         ['sector']),
], remainder='passthrough')

results_target = run_all_models(preprocessor_target)
results_target

## Strategy Comparison Summary

In [ ]:
summary = pd.DataFrame([
    ['1 - Ordinal',          results_ordinal.iloc[0]['Model'], results_ordinal.iloc[0]['CV R2'],  results_ordinal.iloc[0]['Test MAE'],  results_ordinal.iloc[0]['Test RMSE']],
    ['2 - One-Hot (Fixed)',  results_ohe.iloc[0]['Model'],     results_ohe.iloc[0]['CV R2'],      results_ohe.iloc[0]['Test MAE'],      results_ohe.iloc[0]['Test RMSE']],
    ['3 - One-Hot + PCA',    results_pca.iloc[0]['Model'],     results_pca.iloc[0]['CV R2'],      results_pca.iloc[0]['Test MAE'],      results_pca.iloc[0]['Test RMSE']],
    ['4 - Target Encoding',  results_target.iloc[0]['Model'],  results_target.iloc[0]['CV R2'],   results_target.iloc[0]['Test MAE'],   results_target.iloc[0]['Test RMSE']],
], columns=['Strategy', 'Best Model', 'CV R2', 'Test MAE (Cr)', 'Test RMSE (Cr)'])

print(summary.to_string(index=False))
print("\nBest strategy by MAE:", summary.loc[summary['Test MAE (Cr)'].idxmin(), 'Strategy'])

## Hyperparameter Tuning — XGBoost & Random Forest

Original notebook only tuned Random Forest. Since XGBoost scores highest, we tune both and pick the winner.

In [ ]:
# Tune XGBoost with Target Encoding preprocessor
xgb_param_grid = {
    'regressor__n_estimators'    : [200, 300, 500],
    'regressor__max_depth'       : [4, 6, 8],
    'regressor__learning_rate'   : [0.05, 0.1, 0.2],
    'regressor__subsample'       : [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0],
}

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_target),
    ('regressor', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0))
])

xgb_search = GridSearchCV(xgb_pipeline, xgb_param_grid,
                           cv=KFOLD, scoring='r2', n_jobs=-1, verbose=1)
xgb_search.fit(X, y_transformed)
print("XGBoost best params:", xgb_search.best_params_)
print("XGBoost best CV R2 :", round(xgb_search.best_score_, 4))

In [ ]:
# Tune Random Forest with Target Encoding preprocessor
rf_param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth'   : [None, 20, 30],
    # FIX: removed deprecated 'auto' — replaced with 'sqrt' and 0.5
    'regressor__max_features': ['sqrt', 0.5],
    'regressor__max_samples' : [0.5, 0.8, 1.0],
}

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_target),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_search = GridSearchCV(rf_pipeline, rf_param_grid,
                          cv=KFOLD, scoring='r2', n_jobs=-1, verbose=1)
rf_search.fit(X, y_transformed)
print("Random Forest best params:", rf_search.best_params_)
print("Random Forest best CV R2 :", round(rf_search.best_score_, 4))

## Final Model — Select Best & Export

Automatically selects whichever tuned model scored higher, fits on the full dataset, and exports.

In [ ]:
# Auto-select best tuned model
if xgb_search.best_score_ >= rf_search.best_score_:
    best_model_name = 'XGBoost (tuned)'
    final_pipeline  = xgb_search.best_estimator_
    best_score      = xgb_search.best_score_
else:
    best_model_name = 'Random Forest (tuned)'
    final_pipeline  = rf_search.best_estimator_
    best_score      = rf_search.best_score_

print(f"Final model selected : {best_model_name}")
print(f"Best CV R2           : {round(best_score, 4)}")

# Fit on full dataset before export
final_pipeline.fit(X, y_transformed)

with open('pipeline.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)
with open('df.pkl', 'wb') as f:
    pickle.dump(X, f)

print("Saved pipeline.pkl and df.pkl")

## Sanity Check — Sample Prediction

In [ ]:
sample = pd.DataFrame(
    [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'Low', 'Low Floor']],
    columns=X.columns
)

predicted = np.expm1(final_pipeline.predict(sample))[0]
print(f"Sample input  : 4BHK house in Sector 102, 2750 sqft, New, Unfurnished")
print(f"Predicted price: Rs {predicted:.2f} Crores")